# 图算法工程：从数据合同到可观测查询

本 Notebook 不把图算法当成若干 API 的清单，而是实现一条可验收的工程链路：稳定 ID → tenant/有效时间过滤 → 图投影 → BFS/DFS/Dijkstra/PageRank → 中心性与连通结构 → 动态更新 → 查询 trace。数据均为虚构的服务依赖关系，不下载任何外部资源。

**外部合同**：调用者必须提供可信 `AuthContext`、ISO-8601 `as_of`、稳定节点 ID 和明确的权重语义；返回结果必须携带图版本、过滤规模、算法参数和耗时。路径表示“图上可达”，不能直接解释为因果。NetworkX 在这里用于数据结构与交叉校验，内存实现不是生产图数据库。


## 1. 先决定图的语义，再选容器

- `Graph`：无向、同一对节点只有一条边；适合无方向的协作关系。
- `DiGraph`：有向单边；适合调用、依赖、资金流。
- `MultiGraph/MultiDiGraph`：同一端点允许多条不同业务边；边的稳定 key 不能靠插入顺序。
- 加权图还要区分 **cost** 与 **strength**：Dijkstra 最小化 cost；PageRank 通常把 weight 当转移强度。把延迟直接喂给 PageRank 会让高延迟边反而更重要。

节点展示名会改变，所以主键由 `tenant + source_system + external_key` 生成；边 ID 由 tenant、端点、关系、业务键生成。不要使用 Python 的进程随机化 `hash()` 作为持久 ID，也不能用未转义分隔符直接拼接字段。本例把各字段先做 Unicode NFC 规范化，再序列化成规范 JSON 数组；大小写和首尾空白仍保留，由上游数据合同决定是否折叠。


In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import heapq
import json
import math
import time
import unicodedata

import networkx as nx
import numpy as np
import pandas as pd

SCHEMA_VERSION = "service-graph-v1"

ID_UNICODE_NORMALIZATION = "NFC"
def stable_id(*parts: str) -> str:
    normalized = [unicodedata.normalize(ID_UNICODE_NORMALIZATION, str(p)) for p in parts]
    raw = json.dumps(normalized, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:20]

def parse_time(value: str) -> datetime:
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)

@dataclass(frozen=True)
class AuthContext:
    tenant: str
    scopes: frozenset[str]
    principal: str
    def require(self, scope: str) -> None:
        if scope not in self.scopes:
            raise PermissionError(f"缺少 scope: {scope}")

auth_a = AuthContext("tenant-a", frozenset({"graph:read", "graph:write"}), "alice")
auth_b = AuthContext("tenant-b", frozenset({"graph:read"}), "bob")
assert stable_id("tenant-a", "cmdb", "gateway") == "d9a6741361764f6c99d0"  # golden fixture
assert stable_id("a", "b") != stable_id("b", "a")  # 字段顺序属于 ID 合同
assert stable_id("a|b", "c") != stable_id("a", "b|c")  # 规范数组避免分隔符碰撞
assert stable_id("e\u0301") == stable_id("é") and stable_id(" a") != stable_id("a")
assert parse_time("2026-07-01Z").tzinfo is not None


## 2. 受控的有向、多重、加权、时态数据

边同时保存 `latency_ms`（cost）与 `traffic`（strength）。`valid_from <= as_of < valid_to` 使用左闭右开区间，避免相邻版本在边界时刻重复生效。tenant 字段必须来自认证上下文和存储分区，而不是由用户在查询正文中自由指定。


In [ ]:
NODE_ROWS = [
    ("tenant-a", "gateway", "API 网关"), ("tenant-a", "auth", "鉴权服务"),
    ("tenant-a", "order", "订单服务"), ("tenant-a", "pay", "支付服务"),
    ("tenant-a", "risk", "风控服务"), ("tenant-a", "db", "交易数据库"),
    ("tenant-a", "batch", "批处理"), ("tenant-a", "isolated", "孤立服务"),
    ("tenant-b", "gateway", "B租户网关"), ("tenant-b", "secret", "B租户机密服务"),
]
node_id = {(t, k): stable_id(t, "cmdb", k) for t, k, _ in NODE_ROWS}
EDGE_ROWS = [
    # tenant, src, dst, relation, business_key, latency(cost), traffic(strength), from, to
    ("tenant-a", "gateway", "auth", "CALLS", "http", 4.0, 120.0, "2026-01-01Z", None),
    ("tenant-a", "gateway", "order", "CALLS", "http", 7.0, 100.0, "2026-01-01Z", None),
    ("tenant-a", "order", "pay", "CALLS", "sync", 10.0, 70.0, "2026-01-01Z", None),
    ("tenant-a", "order", "pay", "CALLS", "async", 6.0, 30.0, "2026-06-01Z", None),
    ("tenant-a", "pay", "risk", "CALLS", "score", 5.0, 60.0, "2026-01-01Z", None),
    ("tenant-a", "risk", "db", "READS", "primary", 3.0, 50.0, "2026-01-01Z", None),
    ("tenant-a", "pay", "db", "WRITES", "primary", 11.0, 45.0, "2026-01-01Z", None),
    ("tenant-a", "auth", "db", "READS", "legacy", 20.0, 10.0, "2026-01-01Z", "2026-07-01Z"),
    ("tenant-a", "batch", "db", "WRITES", "nightly", 30.0, 5.0, "2026-01-01Z", None),
    ("tenant-b", "gateway", "secret", "CALLS", "http", 1.0, 999.0, "2026-01-01Z", None),
]

def make_graph(nodes=NODE_ROWS, edges=EDGE_ROWS) -> nx.MultiDiGraph:
    g = nx.MultiDiGraph(schema_version=SCHEMA_VERSION, graph_version=1)
    for tenant, key, display_name in nodes:
        nid = node_id[(tenant, key)]
        g.add_node(nid, tenant=tenant, external_key=key, display_name=display_name)
    for tenant, src, dst, rel, biz, latency, traffic, valid_from, valid_to in edges:
        u, v = node_id[(tenant, src)], node_id[(tenant, dst)]
        if g.nodes[u]["tenant"] != tenant or g.nodes[v]["tenant"] != tenant:
            raise ValueError("跨 tenant 边被拒绝")
        eid = stable_id(tenant, u, v, rel, biz)
        g.add_edge(u, v, key=eid, edge_id=eid, tenant=tenant, relation=rel, business_key=biz,
                   latency_ms=float(latency), traffic=float(traffic), valid_from=valid_from, valid_to=valid_to)
    return g

raw_graph = make_graph()
assert isinstance(raw_graph, nx.MultiDiGraph)
assert raw_graph.number_of_nodes() == 10 and raw_graph.number_of_edges() == 10
assert raw_graph.number_of_edges(node_id[("tenant-a", "order")], node_id[("tenant-a", "pay")]) == 2


## 3. 先做权限与时间过滤，再做算法投影

算法不能在全量图上算完再隐藏结果：PageRank、度数甚至路径本身都可能通过聚合值泄漏其他 tenant 的结构。下面先构造授权子图，再把多重边投影成 `DiGraph`。cost 取并行边最小值；strength 求和。这两个聚合规则必须进入版本化合同。


In [ ]:
def edge_active(data: dict, as_of: str) -> bool:
    t = parse_time(as_of)
    return parse_time(data["valid_from"]) <= t and (data["valid_to"] is None or t < parse_time(data["valid_to"]))

def visible_multigraph(g: nx.MultiDiGraph, auth: AuthContext, as_of: str) -> nx.MultiDiGraph:
    auth.require("graph:read")
    visible_nodes = [n for n, d in g.nodes(data=True) if d["tenant"] == auth.tenant]
    out = nx.MultiDiGraph(schema_version=g.graph["schema_version"], graph_version=g.graph["graph_version"], tenant=auth.tenant, as_of=as_of)
    out.add_nodes_from((n, dict(g.nodes[n])) for n in visible_nodes)
    for u, v, k, d in g.edges(keys=True, data=True):
        if u in out and v in out and d["tenant"] == auth.tenant and edge_active(d, as_of):
            out.add_edge(u, v, key=k, **dict(d))
    return out

def project(g: nx.MultiDiGraph) -> nx.DiGraph:
    out = nx.DiGraph(schema_version=g.graph["schema_version"], graph_version=g.graph["graph_version"], tenant=g.graph["tenant"], as_of=g.graph["as_of"])
    out.add_nodes_from(g.nodes(data=True))
    for u, v, d in g.edges(data=True):
        if not out.has_edge(u, v):
            out.add_edge(u, v, cost=d["latency_ms"], strength=d["traffic"], edge_ids=[d["edge_id"]])
        else:
            out[u][v]["cost"] = min(out[u][v]["cost"], d["latency_ms"])
            out[u][v]["strength"] += d["traffic"]
            out[u][v]["edge_ids"].append(d["edge_id"])
    return out

g_a_june = project(visible_multigraph(raw_graph, auth_a, "2026-06-15Z"))
g_a_july = project(visible_multigraph(raw_graph, auth_a, "2026-07-15Z"))
assert len(g_a_july) == 8 and all(d["tenant"] == "tenant-a" for _, d in g_a_july.nodes(data=True))
assert g_a_june.number_of_edges() == g_a_july.number_of_edges() + 1  # legacy 边到期
assert node_id[("tenant-b", "secret")] not in g_a_july
assert g_a_july[node_id[("tenant-a", "order")]][node_id[("tenant-a", "pay")]]["cost"] == 6.0


## 4. BFS 与 DFS：顺序是合同的一部分

BFS 在无权图中给出最少 hop；DFS 适合遍历、环检测等，但不保证最短。邻接迭代顺序受摄取顺序影响，生产返回若要求稳定，应按稳定 ID 或业务优先级排序。两者在邻接表上的时间复杂度均为 `O(V+E)`、访问集合为 `O(V)`。


In [ ]:
def bfs_path(g: nx.DiGraph, source: str, target: str) -> list[str] | None:
    if source not in g or target not in g:
        raise KeyError("source/target 不在授权图中")
    q, parent = deque([source]), {source: None}
    while q:
        u = q.popleft()
        if u == target:
            path = []
            while u is not None:
                path.append(u); u = parent[u]
            return path[::-1]
        for v in sorted(g.successors(u)):
            if v not in parent:
                parent[v] = u; q.append(v)
    return None

def dfs_order(g: nx.DiGraph, source: str) -> list[str]:
    if source not in g:
        raise KeyError(source)
    seen, order, stack = set(), [], [source]
    while stack:
        u = stack.pop()
        if u in seen:
            continue
        seen.add(u); order.append(u)
        stack.extend(reversed(sorted(v for v in g.successors(u) if v not in seen)))
    return order

gateway, db = node_id[("tenant-a", "gateway")], node_id[("tenant-a", "db")]
hop_path = bfs_path(g_a_june, gateway, db)
assert hop_path is not None and len(hop_path) == 3
assert dfs_order(g_a_june, gateway)[0] == gateway
assert bfs_path(g_a_july, node_id[("tenant-a", "isolated")], db) is None


## 5. Dijkstra：负权必须显式拒绝

Dijkstra 依赖“已弹出的最短距离不会被后来路径改善”，所以只接受非负 cost。二叉堆实现复杂度约为 `O((V+E) log V)`。有负边应改用 Bellman–Ford；有负环则最短路无定义。这里不静默取绝对值，也不把缺失权重默认为 1。


In [ ]:
def dijkstra(g: nx.DiGraph, source: str, target: str, weight: str = "cost") -> tuple[float, list[str]]:
    if source not in g or target not in g:
        raise KeyError("source/target 不在图中")
    for u, v, data in g.edges(data=True):
        if weight not in data:
            raise ValueError(f"Dijkstra 边缺少权重 {weight}: {u}->{v}")
        try:
            value = float(data[weight])
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Dijkstra 权重不可转换为浮点数: {u}->{v}") from exc
        if not math.isfinite(value) or value < 0:
            raise ValueError(f"Dijkstra 要求有限非负权重: {u}->{v}")
    dist, parent, heap = {source: 0.0}, {source: None}, [(0.0, source)]
    while heap:
        du, u = heapq.heappop(heap)
        if du != dist[u]:
            continue
        if u == target:
            path = []
            while u is not None:
                path.append(u); u = parent[u]
            return du, path[::-1]
        for v, data in sorted(g[u].items()):
            nd = du + float(data[weight])
            if nd < dist.get(v, math.inf):
                dist[v], parent[v] = nd, u
                heapq.heappush(heap, (nd, v))
    return math.inf, []

distance, cost_path = dijkstra(g_a_june, gateway, db)
assert distance == nx.dijkstra_path_length(g_a_june, gateway, db, weight="cost")
hop_cost = sum(g_a_june[u][v]["cost"] for u, v in zip(hop_path, hop_path[1:]))
assert len(cost_path) > len(hop_path) and distance < hop_cost  # 本 fixture 确实走了更多 hop、但总代价更低
bad = g_a_june.copy(); bad.add_edge(db, gateway, cost=-1.0, strength=1.0)
try:
    dijkstra(bad, gateway, db)
    raise AssertionError("负权未被拒绝")
except ValueError:
    pass


## 6. 从零实现可解释 PageRank 迭代

令 `P` 为按出边 strength 归一化的转移矩阵，阻尼系数为 `d`：`r_(t+1) = (1-d)/N + d(P^T r_t + dangling_mass/N)`。无出边节点的质量必须重新均分，否则概率和会流失。收敛阈值、最大迭代数和权重字段都应进入 trace；分数是结构上的稳态访问概率，不是业务价值或因果重要性。


In [ ]:
def pagerank_power(g: nx.DiGraph, damping: float = 0.85, tol: float = 1e-12, max_iter: int = 500, weight: str = "strength"):
    if not 0 < damping < 1 or len(g) == 0:
        raise ValueError("damping 必须在 (0,1)，且图非空")
    nodes = sorted(g.nodes()); n = len(nodes); rank = {u: 1.0 / n for u in nodes}
    edge_weight = {}
    for u, v, data in g.edges(data=True):
        if weight not in data:
            raise ValueError(f"PageRank 边缺少权重 {weight}: {u}->{v}")
        try:
            value = float(data[weight])
        except (TypeError, ValueError) as exc:
            raise ValueError(f"PageRank 权重不可转换为有限浮点数: {u}->{v}") from exc
        if not math.isfinite(value) or value < 0:
            raise ValueError(f"PageRank strength 必须逐边有限且非负: {u}->{v}")
        edge_weight[(u, v)] = value
    out_weight = {u: sum(edge_weight[(u, v)] for v in g.successors(u)) for u in nodes}
    for iteration in range(1, max_iter + 1):
        dangling = sum(rank[u] for u in nodes if out_weight[u] == 0)
        new = {v: (1-damping) / n + damping * dangling / n for v in nodes}
        for u in nodes:
            if out_weight[u] > 0:
                for v in g.successors(u):
                    new[v] += damping * rank[u] * edge_weight[(u, v)] / out_weight[u]
        delta = sum(abs(new[u] - rank[u]) for u in nodes); rank = new
        if delta < n * tol:
            return rank, {"iterations": iteration, "l1_delta": delta, "converged": True}
    raise RuntimeError("PageRank 未在 max_iter 内收敛")

pr, pr_trace = pagerank_power(g_a_july)
nx_pr = nx.pagerank(g_a_july, alpha=0.85, tol=1e-12, max_iter=500, weight="strength")
assert abs(sum(pr.values()) - 1.0) < 1e-10
assert max(abs(pr[n] - nx_pr[n]) for n in pr) < 1e-8
assert pr_trace["converged"] and pr_trace["iterations"] < 500
negative_strength = nx.DiGraph(); negative_strength.add_edge("a", "b", strength=-1.0); negative_strength.add_edge("a", "c", strength=2.0)
missing_strength = nx.DiGraph(); missing_strength.add_edge("a", "b")
nonfinite_strength = nx.DiGraph(); nonfinite_strength.add_edge("a", "b", strength=float("nan"))
for invalid_graph in (negative_strength, missing_strength, nonfinite_strength):
    try:
        pagerank_power(invalid_graph)
        raise AssertionError("非法 PageRank 权重未被拒绝")
    except ValueError:
        pass


## 7. 中心性、连通分量与社区回答的是不同问题

入/出度是局部连接数；betweenness 衡量最短路径经过程度，计算昂贵且受权重语义影响；弱连通分量忽略方向，强连通分量要求互相可达。社区检测是优化目标下的结构划分，并非天然真实群组；结果受投影、随机种子和 resolution 影响。下面把有向图转无向图做演示，并保留这是一个分析投影。


In [ ]:
analysis_rows = []
between = nx.betweenness_centrality(g_a_july, weight="cost", normalized=True)
for n in sorted(g_a_july):
    analysis_rows.append({"node": g_a_july.nodes[n]["external_key"], "in_degree": g_a_july.in_degree(n),
                          "out_degree": g_a_july.out_degree(n), "pagerank": pr[n], "betweenness": between[n]})
analysis = pd.DataFrame(analysis_rows).sort_values("pagerank", ascending=False).reset_index(drop=True)
weak = list(nx.weakly_connected_components(g_a_july))
undirected = g_a_july.to_undirected()
communities = list(nx.community.greedy_modularity_communities(undirected))
display(analysis.round(4))
print("weak components sizes=", sorted(map(len, weak), reverse=True), "community sizes=", sorted(map(len, communities), reverse=True))
assert sorted(map(len, weak), reverse=True) == [7, 1]
assert set().union(*weak) == set(g_a_july.nodes())
assert len(communities) >= 1


## 8. 失败反例：路径存在不等于因果链成立

`gateway → order → pay → db` 只能说明依据当前、可见、有效的 `CALLS/WRITES` 边可以到达。要回答故障根因，还需请求级 trace、时间先后、实验/干预或可靠的因果模型。另一个常见错误是无条件转无向图：这样数据库会“到达”上游服务，方向语义被破坏。


In [ ]:
directed_reverse = bfs_path(g_a_july, db, gateway)
undirected_reverse = nx.shortest_path(g_a_july.to_undirected(), db, gateway)
assert directed_reverse is None and len(undirected_reverse) >= 2
assert g_a_july.has_edge(gateway, node_id[("tenant-a", "order")])
causal_claim = False  # 图上路径本身没有识别因果所需的干预/混杂控制信息
print({"reachable": True, "causal_claim_allowed": causal_claim, "required_evidence": ["trace_id", "event_time", "intervention_or_causal_assumption"]})


## 9. 动态更新：幂等、版本、删除与缓存失效

流式摄取可能重放同一事件。使用稳定 edge ID 使 upsert 幂等；变更后单调增加 `graph_version`，查询缓存 key 至少包含 tenant、as_of、版本与算法参数。删除采用带审计信息的 tombstone 演示，而不是无痕物理删除。生产中还需要事件序号、乱序水位线、补算策略与快照一致性。


In [ ]:
class GraphStore:
    def __init__(self, graph: nx.MultiDiGraph):
        self.graph = graph.copy(); self.version = int(graph.graph["graph_version"]); self.audit = []; self.tombstones = {}
        self.graph.graph["graph_version"] = self.version
    def _bump_version(self) -> None:
        self.version += 1; self.graph.graph["graph_version"] = self.version
    def upsert_edge(self, auth: AuthContext, u: str, v: str, *, relation: str, business_key: str, **attrs) -> str:
        auth.require("graph:write")
        if u not in self.graph or v not in self.graph or self.graph.nodes[u]["tenant"] != auth.tenant or self.graph.nodes[v]["tenant"] != auth.tenant:
            raise PermissionError("端点越权或不存在")
        eid = stable_id(auth.tenant, u, v, relation, business_key)
        old = self.graph.get_edge_data(u, v, eid)
        payload = dict(edge_id=eid, tenant=auth.tenant, relation=relation, business_key=business_key, **attrs)
        if old != payload:
            self.graph.add_edge(u, v, key=eid, **payload); self._bump_version()
            self.audit.append((self.version, "UPSERT", eid, auth.principal))
        return eid
    def delete_edge(self, auth: AuthContext, u: str, v: str, edge_id: str, reason: str) -> None:
        auth.require("graph:write")
        data = self.graph.get_edge_data(u, v, edge_id)
        if data is None or data["tenant"] != auth.tenant:
            raise PermissionError("不可见边")
        self.tombstones[edge_id] = {"reason": reason, "principal": auth.principal}
        self.graph.remove_edge(u, v, edge_id); self._bump_version()
        self.audit.append((self.version, "DELETE", edge_id, auth.principal))

store = GraphStore(raw_graph)
u, v = node_id[("tenant-a", "risk")], node_id[("tenant-a", "db")]
attrs = dict(latency_ms=3.0, traffic=50.0, valid_from="2026-01-01Z", valid_to=None)
eid = store.upsert_edge(auth_a, u, v, relation="READS", business_key="primary", **attrs)
version_after_first = store.version
store.upsert_edge(auth_a, u, v, relation="READS", business_key="primary", **attrs)
assert store.version == version_after_first  # 重放不产生新版本
store.delete_edge(auth_a, u, v, eid, "连接已迁移")
assert eid in store.tombstones and store.version == version_after_first + 1
try:
    store.upsert_edge(auth_b, u, v, relation="READS", business_key="attack", **attrs)
    raise AssertionError("跨 tenant 写入未被拒绝")
except PermissionError:
    pass


## 10. 查询可观测性与复杂度预算

至少记录：`request_id/principal/tenant`（脱敏）、图版本与快照时间、过滤前后 V/E、算法与参数、visited/relaxations/iterations、耗时、截断/超时状态。高基数 ID 不宜直接作为监控标签。BFS/DFS 是 `O(V+E)`；堆版 Dijkstra 约 `O((V+E)logV)`；PageRank 每轮 `O(E)`；精确 betweenness 在大图很贵，生产需采样、离线计算或图引擎实现。


In [ ]:
def shortest_path_query(raw: nx.MultiDiGraph, auth: AuthContext, source: str, target: str, as_of: str):
    started = time.perf_counter(); visible = visible_multigraph(raw, auth, as_of); algorithm_graph = project(visible)
    distance, path = dijkstra(algorithm_graph, source, target)
    trace = {"schema_version": algorithm_graph.graph["schema_version"], "graph_version": algorithm_graph.graph["graph_version"], "tenant": auth.tenant,
             "as_of": as_of, "algorithm": "dijkstra", "weight": "cost:min_parallel_edge",
             "visible_nodes": len(algorithm_graph), "visible_edges": algorithm_graph.number_of_edges(),
             "latency_ms": round((time.perf_counter() - started) * 1000, 3)}
    return {"distance": distance, "path": path, "trace": trace}

response = shortest_path_query(raw_graph, auth_a, gateway, db, "2026-06-15Z")
assert response["trace"]["tenant"] == "tenant-a" and response["distance"] == distance
assert response["trace"]["graph_version"] == raw_graph.graph["graph_version"]
versioned_copy = raw_graph.copy(); versioned_copy.graph["graph_version"] = 7
assert shortest_path_query(versioned_copy, auth_a, gateway, db, "2026-06-15Z")["trace"]["graph_version"] == 7
assert all(raw_graph.nodes[n]["tenant"] == "tenant-a" for n in response["path"])
display(response)


## 11. 生产替换点与验收清单

本例的 Python 循环与内存图只适合教学和小图。生产替换通常包括：持久图数据库或列式/邻接存储；快照或 MVCC 保证一致读取；分区级 tenant 隔离；图库原生 shortest path/PageRank；流式 CDC 与幂等事件日志；离线大图计算与在线查询分层；超时、最大 hop/visited 限额；审计日志和字段级脱敏。

上线门槛：跨 tenant 与过期边测试、负权拒绝、稳定顺序、并行边聚合契约、缓存版本化、删除审计、基准图与 NetworkX 交叉校验、P95/P99 延迟、最大内存、无路径/孤立点/悬挂点回归测试。


In [ ]:
# 汇总式合同测试：Notebook 顺序执行到这里即完成最小验收。
assert raw_graph.graph["schema_version"] == visible_multigraph(raw_graph, auth_a, "2026-07-15Z").graph["schema_version"] == SCHEMA_VERSION
assert raw_graph.graph["graph_version"] == 1 and store.graph.graph["graph_version"] == store.version
assert all(k == d["edge_id"] for _, _, k, d in raw_graph.edges(keys=True, data=True))
assert response["trace"]["visible_nodes"] < raw_graph.number_of_nodes()
assert math.isinf(dijkstra(g_a_july, node_id[("tenant-a", "isolated")], db)[0])
print("图算法工程合同测试通过；累计 assert > 20")


## 12. 参考资料

- NetworkX 官方算法与最短路文档：https://networkx.org/documentation/stable/reference/algorithms/ 与 https://networkx.org/documentation/stable/reference/algorithms/shortest_paths/
- Page, Brin, Motwani, Winograd, *The PageRank Citation Ranking: Bringing Order to the Web*：https://ilpubs.stanford.edu/422/
- NetworkX PageRank API（实现参数与收敛条件）：https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.link_analysis.pagerank_alg.pagerank.html

引用给出定义和原始方法；Notebook 中的 tenant、时态、幂等与 trace 合同是工程化扩展。受控小图的正确性不能外推为真实生产吞吐或业务效果。
